# Training

In [3]:
import os
os.chdir("../")
os.getcwd()

'/scratch/pawsey1172/sasunih/DeepSpot'

In [4]:
from deepspot.utils.utils_image import get_morphology_model_and_preprocess
from deepspot.utils.utils import plot_loss_values

from deepspot.spot import DeepSpotDataLoader
from deepspot.spot import DeepSpot


from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from pathlib import Path
import lightning as L
import pandas as pd
import numpy as np
import torch
import yaml
yaml.Dumper.ignore_aliases = lambda *args : True

/scratch/pawsey1172/sasunih/miniconda3/envs/deepspot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
samples = set(["KC1"])
out_folder = "TLS_VISIUM_USZ"

In [7]:
dataloader_param = {
# specify the used foundation model 
# to extract the precomputed tile representations
"morphology_model_name": "inception", 
"batch_size": 1024,
# use the spot, subspots, and neighboring spots.
'spot_context': 'spot_subspot_neighbors',
# the radius used to compute the neighbors around 
# the central spot based on the array coordinates.
'radius_neighbors': 1, 
# oversampling
'resolution': 1,
# if to normalize the data during training 
# and the type of normalization
'normalize': 'standard', # None
'augmentation': 'default' # to use 'aestetik' -> pip install aestetik;
        }
batch_size = dataloader_param["batch_size"]
image_feature_model = dataloader_param['morphology_model_name']
num_workers = max(1, torch.get_num_threads() - 1)

del dataloader_param["batch_size"]